<a href="https://colab.research.google.com/github/sylvestersakyi/AToM-OpenMM/blob/rbfe-mutation-workflow/example-notebooks/atom_openmm_protein_peptide_rbfe.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Example of Protein-peptide Alchemical Relative Binding Free Energy Calculation with Atom-OpenMM

### Acknowledgements

- OpenMM: https://openmm.org
- openmmforcefields: https://github.com/openmm/openmmforcefields
- Eric Chen's ATM implementation: https://github.com/EricChen521/atm
- Making-it-rain notebooks:  https://github.com/pablo-arantes/making-it-rain
- TIAM1/Syndecan-1 system: https://doi.org/10.1021/acs.jcim.5c00207
- Thanks to Stefan Doerr @Acellera for helping modernizing the atom-openmm modules.

### Procedure

- In this example, the pdb file of a protein receptor (TIAM1) and the pdb files of two ligands already docked to the receptor are combined into a complex.
- The second ligand is displaced into the solvent by an automatically chosen displacement vector.
- The two ligands are aligned using the backbone CA, N, and C atoms of the mutation residue.
- The system is solvated, minimized, thermalized, and annealed at the alchemical intermediate (when the two ligands are half bound and half unbound).
- Then an alchemical Hamiltonian replica exchange simulation is carried out.
- Finally, the perturbation energy data collected during the simulation is analyzed to obtain the relative binding free energy of the two ligands.

Read the [introduction to AToM-OpenMM](https://www.compmolbiophysbc.org/atom-openmm) to get a better sense of steps involved.


### To run the notebook

- Connect to a Google Colab Runtime by clicking on the down-arrow next to the `Connect` button on the upper left, select `Change runtime type`, and selecting a `T4` GPU. Then click `Connect`.

- Execute the first cell to install `condacolab`. The runtime will restart. When done, move to the following cells.

- The final cells helps you download the simulation results.


### **Install Conda Colab (if on Google Colab)**

It will restart the kernel (it's okay, the warning is harmless)


In [1]:
!pip install -q condacolab
import condacolab
condacolab.install_from_url("https://github.com/conda-forge/miniforge/releases/download/25.3.1-0/Miniforge3-Linux-x86_64.sh")


⏬ Downloading https://github.com/conda-forge/miniforge/releases/download/25.3.1-0/Miniforge3-Linux-x86_64.sh...
📦 Installing...
📌 Adjusting configuration...
🩹 Patching environment...
⏲ Done in 0:00:12
🔁 Restarting kernel...


### **Install dependencies**


In [2]:
import subprocess
import sys

#fixes sys.path that gives torchvision import error
original_syspath = sys.path.copy()
new_syspath = ['/content', '/env/python', '/usr/local/lib/python312.zip', '/usr/local/lib/python3.12', '/usr/local/lib/python3.12/lib-dynload', '/usr/local/lib/python3.12/site-packages' ]
sys.path = new_syspath

#dependencies from conda-forge
subprocess.run("mamba install openmm openmmforcefields setproctitle -y", shell=True)

#latest atom-openmm (or 'pip install atom-openmm' for the latest released version)
#subprocess.run("cd /content && git clone https://github.com/Gallicchio-Lab/AToM-OpenMM.git", shell=True)
#subprocess.run("cd /content/AToM-OpenMM && pip install .", shell=True)

#CHANGE-ME: download from development branch. Change to the above once the PR is included into the main repository
subprocess.run("cd /content && git clone -b rbfe-mutation-workflow https://github.com/sylvestersakyi/AToM-OpenMM.git", shell=True)
subprocess.run("cd /content/AToM-OpenMM && pip install .", shell=True)

#installs py3Dmol
!pip install py3Dmol


### **Test the OpenMM installation**


In [1]:
import openmm.testInstallation
openmm.testInstallation.main()



OpenMM Version: 8.5.1
Git Revision: f7fa0c27c1f8d943c339d67b3bf22f026d0bd8b5

There are 4 Platforms available:

1 Reference - Successfully computed forces
2 CPU - Successfully computed forces
3 CUDA - Successfully computed forces
4 OpenCL - Successfully computed forces

Median difference in forces between platforms:

Reference vs. CPU: 6.29154e-06
Reference vs. CUDA: 6.74871e-06
CPU vs. CUDA: 7.62776e-07
Reference vs. OpenCL: 6.74321e-06
CPU vs. OpenCL: 7.88789e-07
CUDA vs. OpenCL: 1.77022e-07

All differences are within tolerance.


### **Prepare the ATM system in a box of water**

It requires the pdb file of the receptor and the pdb files of the ligands already docked to the receptor.

In this example, they are taken from the ATom-OpenMM repo. In general, you might want to upload them somewhere in the Colab instance.

It uses the basename of the receptor and ligand file names to contruct the job name (for example `tiam1-sdc1wt-sdc1Q3E`). The job working directory is taken as `/content/` followed by the job name.

The initial position of the second ligand in the solvent is selected automatically to minimize the simulation box size.


In [4]:
import os
from atom_openmm.make_atm_system_from_rcpt_lig import make_system
from atom_openmm.utils.AtomUtils import calc_displ_vec
from pathlib import Path

#collect names of receptor pdb file and ligand pdb files
Protein_PDB_file_name = 'tiam1.pdb' #@param {type:"string"}
Ligand1_PDB_file_name = 'sdc1wt.pdb' #@param {type:"string"}
Ligand2_PDB_file_name = 'sdc1Q3E.pdb' #@param {type:"string"}
Mutation_residue_number = 3 #@param {type:"integer"}
Receptor_chain_name = 'B' #@param {type:"string"}

#where the receptor and ligand file name are stored
dataDir = '/content/AToM-OpenMM/example-notebooks/data'


#full file names
rcpt_pdb = dataDir + '/' + Protein_PDB_file_name
ligand1_pdb = dataDir + '/' + Ligand1_PDB_file_name
ligand2_pdb = dataDir + '/' + Ligand2_PDB_file_name

#automatic selection of displacement vector
#adapted from Eric Chen's ATM implementation: https://github.com/EricChen521/atm
Displacement_vector_in_Angstroms = list(calc_displ_vec(rcpt_pdb, ligand2_pdb))
print("Displacement vector:", Displacement_vector_in_Angstroms, " Angstroms" )

#basename of the job (the job name)
basename = Path(Protein_PDB_file_name).stem + \
  '-' + \
  Path(Ligand1_PDB_file_name).stem + \
  '-' + \
  Path(Ligand2_PDB_file_name).stem \

#create the working directory and change to it
workDir = f'/content/{basename}'
os.makedirs(workDir, exist_ok=True)
os.chdir(workDir)

#the ATM system xml file and the output pdb file
outxml = workDir + '/' + basename + '_sys.xml'
pdboutfile = workDir + '/' + basename + '.pdb'

#builds the system
make_system(
      receptorfile=rcpt_pdb,
      lig1file=ligand1_pdb,
      lig2file=ligand2_pdb,
      displacement = Displacement_vector_in_Angstroms,
      xmloutfile=outxml,
      pdboutfile=pdboutfile,
      ffcachefile='ff.json',
      hmass=1.5,
)


AttributeError: partially initialized module 'torchvision' has no attribute 'extension' (most likely due to a circular import)

### **View the Protein-Ligand Complex in water**

The second ligand is displaced from the first in the solvent by the displacement vector.

Try a different displacement vector if this location is not satisfactory. For example, if it is too close to the receptor or, alternatively, if it leads to an excessively large simulation box.


In [ ]:
import py3Dmol

with open(pdboutfile, "r") as f:
    pdb_data = f.read()

view = py3Dmol.view(width=800, height=600)
view.addModel(pdb_data, "pdb")

# Show Protein as Cartoon
view.setStyle({'chain': Receptor_chain_name}, {'cartoon': {'color': 'spectrum'}})

# Show Ligands as Sticks
view.setStyle({'chain': 'L'}, {'stick': {'color': 'cyan'}})
view.setStyle({'chain': 'M'}, {'stick': {'color': 'orange'}})

# Hide Solvent
view.setStyle({'resn': ['SOL', 'WAT', 'HOH', 'TIP3']}, {})

view.zoomTo()
view.show()

ModuleNotFoundError: No module named 'py3Dmol'

### **Set the Ligand Alignment Atoms**

ATM requires alignment of the coordinate frames of the two ligands. This is achieved by identifying three corresponding non-colinear atoms of each ligand. For this example, these are the backbone CA, N, and C atoms of the mutation residue.

You can read more about alignment atoms [here](https://www.compmolbiophysbc.org/atom-openmm/atom-system-setup#h.vndnoxipr7qs).

Look at the structure of each ligand below and verify the automatically selected reference atoms for these specific ligands.


In [ ]:
#@title ##### **View Ligand 1 to Verify Alignment Atoms**
with open(ligand1_pdb, "r") as f:
    ligand1_data = f.read()

view = py3Dmol.view(width=800, height=600)
view.addModel(ligand1_data, "pdb")

view.setStyle({}, {'cartoon': {'color': 'cyan', 'style': 'tube'}})
view.addStyle({'resi': str(Mutation_residue_number)}, {'stick': {'radius': 0.15}})
view.addPropertyLabels("serial", {'resi': str(Mutation_residue_number)},
                       {"fontColor":"red", "alignment":"center"})
view.zoomTo({'resi': str(Mutation_residue_number)})
view.show()


In [ ]:
def collect_backbone_atom_ids(pdb_file, residue_id):
    atom_order = ('CA', 'N', 'C')
    atom_ids = {}
    with open(pdb_file, 'r') as f:
        for line in f:
            if not line.startswith(('ATOM  ', 'HETATM')):
                continue
            if line[22:26].strip() != str(residue_id):
                continue
            atom_name = line[12:16].strip()
            if atom_name in atom_order and atom_name not in atom_ids:
                atom_ids[atom_name] = int(line[6:11])
    missing = [name for name in atom_order if name not in atom_ids]
    if missing:
        raise ValueError(f'{pdb_file}: residue {residue_id} is missing atoms {missing}')
    return [atom_ids[name] for name in atom_order]

view = py3Dmol.view(width=800, height=600)
view.addModel(ligand1_data, "pdb")

view.setStyle({}, {'cartoon': {'color': 'cyan', 'style': 'tube'}})
view.addStyle({'resi': str(Mutation_residue_number)}, {'stick': {'radius': 0.15}})

ligand1_ref_atoms = collect_backbone_atom_ids(ligand1_pdb, Mutation_residue_number)

view.addStyle({'serial': ligand1_ref_atoms}, {'sphere': {'radius': 0.4}})

view.zoomTo({'serial': ligand1_ref_atoms})
view.show()
print('Ligand 1 alignment atom ids (CA, N, C):', ligand1_ref_atoms)


In [ ]:
#@title ##### **View Ligand 2 to Verify Alignment Atoms**
with open(ligand2_pdb, "r") as f:
    ligand2_data = f.read()

view = py3Dmol.view(width=800, height=600)
view.addModel(ligand2_data, "pdb")

view.setStyle({}, {'cartoon': {'color': 'orange', 'style': 'tube'}})
view.addStyle({'resi': str(Mutation_residue_number)}, {'stick': {'radius': 0.15}})
view.addPropertyLabels("serial", {'resi': str(Mutation_residue_number)},
                       {"fontColor":"red", "alignment":"center"})
view.zoomTo({'resi': str(Mutation_residue_number)})
view.show()


In [ ]:
view = py3Dmol.view(width=800, height=600)
view.addModel(ligand2_data, "pdb")

view.setStyle({}, {'cartoon': {'color': 'orange', 'style': 'tube'}})
view.addStyle({'resi': str(Mutation_residue_number)}, {'stick': {'radius': 0.15}})

ligand2_ref_atoms = collect_backbone_atom_ids(ligand2_pdb, Mutation_residue_number)

view.addStyle({'serial': ligand2_ref_atoms}, {'sphere': {'radius': 0.4}})

view.zoomTo({'serial': ligand2_ref_atoms})
view.show()
print('Ligand 2 alignment atom ids (CA, N, C):', ligand2_ref_atoms)
print('Auto-selected alignment atom ids:')
print('  Ligand 1:', ligand1_ref_atoms)
print('  Ligand 2:', ligand2_ref_atoms)


### **Setup Relative Binding Free Energy Calculation**

It does energy minimization, thermalization, equilibration, and annealing to the alchemical intermediate at lambda=1/2

It takes about 1/2 hour on a T4 GPU.


In [ ]:
import os
import openmm as mm
from openmm.app import *
from openmm import *
from openmm.unit import *
from multiprocessing import freeze_support
from atom_openmm.rbfe_structprep import rbfe_structprep
from atom_openmm.rbfe_production import rbfe_production
from atom_openmm.utils.AtomUtils import AtomUtils, get_selected_principal_groups, get_indexes_from_query, get_indexes_from_residue, cm_from_indexes

#common atoms are the ligand atoms outside of the variable sidechains
def commonatoms(topology, lig1_atoms, varatoms1, lig2_atoms, varatoms2):
    ca1 = sorted(list(set(lig1_atoms) - set(varatoms1)))
    ca2 = sorted(list(set(lig2_atoms) - set(varatoms2)))
    commonatoms1 = []
    commonatoms2 = []
    atoms = list(topology.atoms())

    for i in ca1:
        found = False
        for j in ca2:
            if atoms[i].residue.id == atoms[j].residue.id and atoms[i].name == atoms[j].name:
                commonatoms1.append(i)
                commonatoms2.append(j)
                found = True
                break
        if not found:
            raise ValueError(f'No matching atom for atom {i} ({atoms[i].residue.id}:{atoms[i].name})')
    if len(ca1) != len(commonatoms1) or len(ca2) != len(commonatoms2):
        raise ValueError('common atoms do not match')
    return commonatoms1, commonatoms2

options = {
        "JOB_TRANSPORT": 'LOCAL_OPENMM',
        "RE_SETUP": 'YES',
        "TEMPERATURES": [ 310.0 ],
        "LAMBDAS":      [0.00, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90, 0.95, 1.00],
        "DIRECTION":    [   1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,   -1,   -1,   -1,   -1,   -1,   -1,   -1,   -1,   -1,   -1,   -1],
        "INTERMEDIATE": [   0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    1,    1,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0],
        "LAMBDA1":      [0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.10, 0.20, 0.30, 0.40, 0.50, 0.50, 0.40, 0.30, 0.20, 0.10, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00],
        "LAMBDA2":      [0.00, 0.10, 0.20, 0.30, 0.40, 0.50, 0.50, 0.50, 0.50, 0.50, 0.50, 0.50, 0.50, 0.50, 0.50, 0.50, 0.50, 0.40, 0.30, 0.20, 0.10, 0.00],
        "ALPHA":        [0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10],
        "U0":           [110., 110., 110., 110., 110., 110., 110., 110., 110., 110., 110., 110., 110., 110., 110., 110., 110., 110., 110., 110., 110., 110.],
        "W0COEFF":      [   0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0],
        "WALL_TIME": 240, #max simulation time in minutes
        "MAX_SAMPLES": 10, #number of samples/replica to collect
        "CYCLE_TIME": 10, #interval between replica-exchange exchanges in seconds
        "CHECKPOINT_TIME": 1200, #interval between checkpoints in seconds
        "SUBJOBS_BUFFER_SIZE": 1.0, #fraction of replicas to keep in fast execution queue, in units of number of GPU devices
        "PRODUCTION_STEPS": 10000, #how many MD steps to run each replica
        "PRNT_FREQUENCY":   10000, #interval between printing replica data, in MD steps
        "TRJ_FREQUENCY":    20000, #interval between trajectory frame records, in MD steps
        "CM_KF": 2.5, #force constant of flat-bottom binding site restraint, in kcal/mol/Ang^2
        "CM_TOL": 7.5, #tolerance of flat-bottom binding site restraint, radius in Ang
        "POSRE_FORCE_CONSTANT": 25.0, #force constant of position restraint, in kcal/mol/Ang^2
        "POSRE_TOLERANCE": 2.5, #tolerance of position restraint, radius in Ang
        "ALIGN_KF_SEP": 0.0, #force constant of separation component of the ligand alignment restraint, in kcal/mol/Ang^2
        "ALIGN_K_THETA": 25.0, #force constant of orientation component of the ligand alignment restraint, in kcal/mol
        "ALIGN_K_PSI":   25.0, #force constant of roll component of the ligand alignment restraint, in kcal/mol
        "UMAX": 200.00, #max energy parameter of the soft-core binding energy, in kcal/mol
        "ACORE": 0.062500, #a-parameter of the soft-core binding energy, dimensionless
        "UBCORE": 100.0, #ub-parameter of the soft-core binding energy, in kcal/mol
        "FRICTION_COEFF": 0.500000, #temperature friction relaxation time, in ps
        "HMASS": 1.5, #hydrogen mass for mass repartitioning
        "TIME_STEP": 0.002, #MD time-step in ps
        "VERBOSE": False,
  }

#job basename
options['BASENAME'] = basename

#displacement
options['DISPLACEMENT'] = Displacement_vector_in_Angstroms

#system pdb file
pdb = PDBFile(pdboutfile)
topology = pdb.topology
positions = pdb.positions

#protein backbone atom names
backbone = """["N","NT", "CA","CAY", "CAT", "C", "CY", "O", "OY", "OXT", "H", "H1", "H2", "H3", "HT1", "HT2", "HT3", "HNT", "HY1", "HY2", "HY3", "HA", "DN", "DCA", "DC", "DO", "LPOA", "LPOB",  "DOT1", "DOT2", "LPT1", "LPT2", "LPT3", "LPT4"]"""

#LIGAND1_ATOMS (chain L)
lig1chain = None
for chain in topology.chains():
    if chain.id == 'L':
        lig1chain = chain
        break
assert(lig1chain)
ligand1_atoms = sorted([atom.index for atom in lig1chain.atoms()])
options['LIGAND1_ATOMS'] = ligand1_atoms

#LIGAND2_ATOMS (chain M)
lig2chain = None
for chain in topology.chains():
    if chain.id == 'M':
        lig2chain = chain
        break
assert(lig2chain)
ligand2_atoms = sorted([atom.index for atom in lig2chain.atoms()])
options['LIGAND2_ATOMS'] = ligand2_atoms

#mutation residue on each ligand
res1 = None
for residue in lig1chain.residues():
    if residue.id == str(Mutation_residue_number):
        res1 = residue
        break
assert(res1)
res2 = None
for residue in lig2chain.residues():
    if residue.id == str(Mutation_residue_number):
        res2 = residue
        break
assert(res2)

#variable sidechain atoms
options['LIGAND1_VAR_ATOMS'] = get_indexes_from_residue(res1, f'not atom.name in {backbone}')
options['LIGAND2_VAR_ATOMS'] = get_indexes_from_residue(res2, f'not atom.name in {backbone}')

#common atoms of the two ligands
common1, common2 = commonatoms(topology,
                               options['LIGAND1_ATOMS'],
                               options['LIGAND1_VAR_ATOMS'],
                               options['LIGAND2_ATOMS'],
                               options['LIGAND2_VAR_ATOMS'])
options['LIGAND1_COMMON_ATOMS'] = common1
options['LIGAND2_COMMON_ATOMS'] = common2

#alignment atoms from the mutation residue CA, N, C
ligand1_ref_atoms = []
ligand2_ref_atoms = []
for atom_name in ['CA', 'N', 'C']:
    lig1_matches = get_indexes_from_query(topology, f'atom.residue.chain.id == "L" and atom.residue.id == "{Mutation_residue_number}" and atom.name == "{atom_name}"')
    lig2_matches = get_indexes_from_query(topology, f'atom.residue.chain.id == "M" and atom.residue.id == "{Mutation_residue_number}" and atom.name == "{atom_name}"')
    if not lig1_matches or not lig2_matches:
        raise ValueError(f'Missing alignment atom {atom_name} at residue {Mutation_residue_number}')
    ligand1_ref_atoms.append(lig1_matches[0] - ligand1_atoms[0])
    ligand2_ref_atoms.append(lig2_matches[0] - ligand2_atoms[0])
options['ALIGN_LIGAND1_REF_ATOMS'] = ligand1_ref_atoms
options['ALIGN_LIGAND2_REF_ATOMS'] = ligand2_ref_atoms

#attachment atoms (first reference atom)
options['LIGAND1_ATTACH_ATOM'] = ligand1_atoms[ligand1_ref_atoms[0]]
options['LIGAND2_ATTACH_ATOM'] = ligand2_atoms[ligand2_ref_atoms[0]]

#cm atom of the ligand is the first reference atom
options['LIGAND1_CM_ATOMS'] = [ ligand1_atoms[ligand1_ref_atoms[0]] ]
options['LIGAND2_CM_ATOMS'] = [ ligand2_atoms[ligand2_ref_atoms[0]] ]

#CMs of the ligands
lig1cm = cm_from_indexes(topology, positions, options['LIGAND1_CM_ATOMS'] )
lig2cm = cm_from_indexes(topology, positions, options['LIGAND2_CM_ATOMS'] )

#actual displacement between the ligand CMs
displ = (lig2cm - lig1cm).value_in_unit(angstrom)
options['DISPLACEMENT'] = [ displ.x , displ.y, displ.z ]

#CM atoms of the receptor, all C-alpha atoms
query = f'atom.residue.chain.id == "{Receptor_chain_name}" and atom.name == "CA"'
rcpt_frame_indexes = get_indexes_from_query(topology, query)

#internal receptor frame
rcpt_frame = get_selected_principal_groups(topology, positions, rcpt_frame_indexes)

#receptor CM atoms (origin of the internal frame)
options['RCPT_CM_ATOMS'] = rcpt_frame['origin']['indices']
options['RCPT_FRAME_ATOMS_O'] = options['RCPT_CM_ATOMS']
options['RCPT_FRAME_ATOMS_Z'] = rcpt_frame['z_axis']['indices']
options['RCPT_FRAME_ATOMS_Y'] = rcpt_frame['y_axis']['indices']

#offset is the distance from the CM of the first ligand from the CM of the receptor
rcpt_cm_pos = Vec3(float(rcpt_frame['origin']['com'][0]),
                   float(rcpt_frame['origin']['com'][1]),
                   float(rcpt_frame['origin']['com'][2])) * nanometer
offset = (lig1cm - rcpt_cm_pos).value_in_unit(angstrom)
options['LIGOFFSET'] = [ offset.x, offset.y, offset.z ]

#restrained atoms (same as RCPT_CM_ATOMS)
options['POS_RESTRAINED_ATOMS'] = options['RCPT_CM_ATOMS']

#receptor-ligand exclusion potential
options['EXCLUSION_POT_MOL1_INDEXES'] = get_indexes_from_query(
    topology, f'(atom.residue.chain.id == "{Receptor_chain_name}") and (atom.residue.name != "HOH") and (atom.element.atomic_number != 1)')
options['EXCLUSION_POT_MOL2_INDEXES'] = get_indexes_from_query(
    topology, '(atom.residue.chain.id == "M") and (atom.element.atomic_number != 1)')

#working directory
options['WORKDIR'] = workDir

print("ATM RBFE SETTINGS:")
print(options)

rbfe_structprep(config_file = None, options = options)


### **Run Relative Binding Free Energy Calculation**

It will take an hour or so, or change `MAX_SAMPLES` or `WALL_TIME` above.


In [ ]:
rbfe_production(config_file = None, options = options)


### **Analyisis: Binding free energy estimate**

Runs UWHAM to get the relative binding free energy estimate. By default, 1/3 of the early data is discarded for equilibration. It would take a much longer to reach a reliable estimate. This is only an example.

The quality assessment plot shows the perturbation energy distributions for the two alchemical legs (first row). Good overlaps between the distributions is a sign of good convergence. However, there aren't enough samples here to make a definitive assessment.

The bottom row of plots shows the lambda function and the derivative of the alchemical perturbation energy function. The intersections between these determines the maxima and minima of the distributions. These graphs help optimize the alchemical schedule. See https://doi.org/10.1063/5.0244918, for example.


In [ ]:
from atom_openmm.uwham import calculate_uwham_from_rundir, create_quality_assessment_plot

ddG, ddG_std, uwham_data = calculate_uwham_from_rundir(
          options['WORKDIR'], options['BASENAME']
)
print(ddG, '+/-', ddG_std, "kcal/mol")

#produces a plot for simulation quality assessment
df1 = uwham_data['df_leg1']
N = len(uwham_data['uwham_out_leg1']['W'][:,0])
df1['W'] = uwham_data['uwham_out_leg1']['W'][:,0]/float(N)
df2 = uwham_data['df_leg2']
N = len(uwham_data['uwham_out_leg2']['W'][:,0])
df2['W'] = uwham_data['uwham_out_leg2']['W'][:,0]/float(N)
fig = create_quality_assessment_plot(df1, df2)


### **Download the results as a zip file**


In [ ]:
from google.colab import files
print(basename)
!cd /content && zip -r  {basename}.zip {basename}
filename = f'/content/{basename}.zip'
files.download(filename)
